In [30]:
import pandas as pd
import sqlite3
import os
def sql(query):
    return display(pd.read_sql(query, conn))

In [31]:
energy = pd.read_csv("energy_cleaned.csv")
temperature = pd.read_csv("temperature_cleaned.csv")

In [32]:
conn = sqlite3.connect("energy_analysis.db")

In [33]:
energy.to_sql("energy", conn, if_exists="replace", index=False)
temperature.to_sql("temperature", conn, if_exists="replace", index=False)

20740

In [34]:
query = "SELECT * FROM energy LIMIT 10"
result = pd.read_sql(query, conn)
display(result)

,Country,Energy Type,Year,Energy Consumption,Energy Production,GDP,Population,Energy Intensity Per Capita,Energy Intensity By GDP,CO2 Emission
0,Afghanistan,all_energy_types,1980-01-01,0.026583,0.072561,47.7571,13356.5,0.001990,0.000557,NaN
1,Afghanistan,coal,1980-01-01,0.002479,0.002355,47.7571,13356.5,0.000186,0.000052,NaN
2,Afghanistan,natural_gas,1980-01-01,0.002094,0.062820,47.7571,13356.5,0.000157,0.000044,NaN
3,Afghanistan,petroleum_n_other_liquids,1980-01-01,0.014624,0.000000,47.7571,13356.5,0.001095,0.000306,NaN
4,Afghanistan,renewables_n_other,1980-01-01,0.007386,0.007386,47.7571,13356.5,0.000553,0.000155,0.0
5,Albania,all_energy_types,1980-01-01,0.162982,0.155562,47.7571,2682.7,0.060753,0.003413,NaN
6,Albania,coal,1980-01-01,0.024317,0.013229,47.7571,2682.7,0.009064,0.000509,NaN
7,Albania,natural_gas,1980-01-01,0.010470,0.010470,47.7571,2682.7,0.003903,0.000219,NaN
8,Albania,petroleum_n_other_liquids,1980-01-01,0.099297,0.101540,47.7571,2682.7,0.037014,0.002079,NaN
9,Albania,renewables_n_other,1980-01-01,0.028897,0.030323,47.7571,2682.7,0.010772,0.000605,0.0


In [35]:
# Check current column names
print(energy.columns.tolist())

['Country', 'Energy Type', 'Year', 'Energy Consumption', 'Energy Production', 'GDP', 'Population', 'Energy Intensity Per Capita', 'Energy Intensity By GDP', 'CO2 Emission']


In [36]:
# Clean them
energy.columns = energy.columns.str.strip().str.replace(" ", "_").str.lower()

In [37]:
# Verify new names
print(energy.columns.tolist())

['country', 'energy_type', 'year', 'energy_consumption', 'energy_production', 'gdp', 'population', 'energy_intensity_per_capita', 'energy_intensity_by_gdp', 'co2_emission']


In [38]:
# Reload into SQL
energy.to_sql("energy", conn, if_exists="replace", index=False)

47640

In [39]:
sql("""
PRAGMA table_info(energy)
""")

,cid,name,type,notnull,dflt_value,pk
0,0,country,TEXT,0,None,0
1,1,energy_type,TEXT,0,None,0
2,2,year,TEXT,0,None,0
3,3,energy_consumption,REAL,0,None,0
4,4,energy_production,REAL,0,None,0
5,5,gdp,REAL,0,None,0
6,6,population,REAL,0,None,0
7,7,energy_intensity_per_capita,REAL,0,None,0
8,8,energy_intensity_by_gdp,REAL,0,None,0
9,9,co2_emission,REAL,0,None,0


In [40]:
sql("""
SELECT country, year, co2_emission
FROM energy
WHERE energy_type = 'coal'
ORDER BY co2_emission DESC
""")


,country,year,co2_emission
0,China,2014-01-01,9113.558191
1,China,2013-01-01,8970.681630
2,China,2015-01-01,8939.781470
3,China,2016-01-01,8646.270696
4,China,2019-01-01,8546.287450
...,...,...,...
9195,Former U.S.S.R.,2019-01-01,NaN
9196,Former Yugoslavia,2019-01-01,NaN
9197,"Germany, East",2019-01-01,NaN
9198,"Germany, West",2019-01-01,NaN


In [41]:
# 1. Which countries have highest CO2 emissions?
energy[energy["energy_type"] == "all_energy_types"].groupby("country")["co2_emission"].mean().sort_values(ascending=False).head(10)

country
United States      5290.096649
China              5071.245114
Former U.S.S.R.    3792.565229
Russia             1676.491372
Japan              1134.363443
India              1023.213320
Germany             858.767597
Germany, West       717.836652
United Kingdom      554.522657
Germany, East       521.636171
Name: co2_emission, dtype: float64

In [42]:
# 2. How has global energy consumption changed over time?
energy[energy["energy_type"] == "all_energy_types"].groupby("year")["energy_consumption"].sum()

year
1980-01-01    369.455863
1981-01-01    365.957095
1982-01-01    366.247175
1983-01-01    370.287699
1984-01-01    383.790439
1985-01-01    391.983221
1986-01-01    398.739473
1987-01-01    409.365512
1988-01-01    422.180143
1989-01-01    427.971623
1990-01-01    430.129478
1991-01-01    432.315695
1992-01-01    437.739291
1993-01-01    440.625021
1994-01-01    444.457850
1995-01-01    454.299409
1996-01-01    464.127049
1997-01-01    465.283074
1998-01-01    467.415076
1999-01-01    474.616058
2000-01-01    487.304857
2001-01-01    490.146496
2002-01-01    498.897045
2003-01-01    513.425381
2004-01-01    535.612891
2005-01-01    550.216855
2006-01-01    565.099860
2007-01-01    578.130800
2008-01-01    584.777819
2009-01-01    580.058902
2010-01-01    611.722288
2011-01-01    625.486947
2012-01-01    636.571837
2013-01-01    646.063605
2014-01-01    651.101396
2015-01-01    652.300276
2016-01-01    657.130685
2017-01-01    669.173270
2018-01-01    682.708036
2019-01-01    686.01

In [43]:
print("Energy columns:", energy.columns.tolist())
print("Temperature columns:", temperature.columns.tolist())

Energy columns: ['country', 'energy_type', 'year', 'energy_consumption', 'energy_production', 'gdp', 'population', 'energy_intensity_per_capita', 'energy_intensity_by_gdp', 'co2_emission']
Temperature columns: ['Country', 'Year', 'Annual_Mean', 'Five_Year_Smooth', 'Code']


In [44]:
temperature.columns = temperature.columns.str.strip().str.replace(" ", "_").str.lower()

In [45]:
# 3. Merge both datasets to see CO2 vs temperature relationship
merged = pd.merge(
    energy[energy["energy_type"] == "all_energy_types"],
    temperature,
    on=["country", "year"],
    how="inner"
)
merged[["country", "year", "co2_emission", "annual_mean"]].head()

,country,year,co2_emission,annual_mean
0,Afghanistan,1980-01-01,NaN,13.01
1,Albania,1980-01-01,NaN,10.97
2,Algeria,1980-01-01,NaN,22.94
3,Angola,1980-01-01,NaN,21.50
4,Argentina,1980-01-01,NaN,15.13


In [46]:
result = merged[
    (merged["country"] == "Afghanistan") & 
    (merged["co2_emission"].notna())
].head(5)

display(result)

,country,energy_type,year,energy_consumption,energy_production,gdp,population,energy_intensity_per_capita,energy_intensity_by_gdp,co2_emission,annual_mean,five_year_smooth,code
148,Afghanistan,all_energy_types,1981-01-01,0.029949,0.092724,47.7571,13171.7,0.002274,0.000627,1.325965,13.06,12.87,AFG
296,Afghanistan,all_energy_types,1982-01-01,0.031898,0.095068,47.7571,12882.5,0.002476,0.000668,1.509628,12.01,12.86,AFG
444,Afghanistan,all_energy_types,1983-01-01,0.039022,0.099713,47.7571,12537.7,0.003112,0.000817,1.659605,12.91,12.82,AFG
592,Afghanistan,all_energy_types,1984-01-01,0.039202,0.099827,47.7571,12204.3,0.003212,0.000821,2.059605,12.50,12.78,AFG
740,Afghanistan,all_energy_types,1985-01-01,0.038275,0.120968,47.7571,11938.2,0.003206,0.000801,2.065436,13.03,12.76,AFG


In [47]:
merged["year"] = pd.to_datetime(merged["year"]).dt.year
display(merged.head())

,country,energy_type,year,energy_consumption,energy_production,gdp,population,energy_intensity_per_capita,energy_intensity_by_gdp,co2_emission,annual_mean,five_year_smooth,code
0,Afghanistan,all_energy_types,1980,0.026583,0.072561,47.7571,13356.50,0.001990,0.000557,NaN,13.01,12.84,AFG
1,Albania,all_energy_types,1980,0.162982,0.155562,47.7571,2682.70,0.060753,0.003413,NaN,10.97,11.46,ALB
2,Algeria,all_energy_types,1980,0.780695,2.803017,47.7571,19221.70,0.040615,0.016347,NaN,22.94,23.07,DZA
3,Angola,all_energy_types,1980,0.058366,0.335098,47.7571,8341.29,0.006997,0.001222,NaN,21.50,21.47,AGO
4,Argentina,all_energy_types,1980,1.713088,1.597946,453.8190,27951.93,0.061287,0.003775,NaN,15.13,15.02,ARG


In [48]:
# Check how many rows merged successfully
print(f"Merged rows: {len(merged)}")
print(f"Countries: {merged['country'].nunique()}")
print(f"Year range: {merged['year'].min()} - {merged['year'].max()}")

Merged rows: 5920
Countries: 148
Year range: 1980 - 2019


In [49]:
print(f"CO2 nulls: {merged['co2_emission'].isna().sum()}")

CO2 nulls: 424


In [50]:
merged.to_csv("merged_final.csv", index=False)

In [51]:
test = pd.read_csv("merged_final.csv")
print(f"Rows: {len(test)}")
print(f"Countries: {test['country'].nunique()}")
print(f"China rows: {len(test[test['country'] == 'China'])}")
print(f"India rows: {len(test[test['country'] == 'India'])}")

Rows: 5920
Countries: 148
China rows: 0
India rows: 0


In [52]:
energy = pd.read_csv("energy_cleaned.csv")
energy.columns = energy.columns.str.strip().str.replace(" ", "_").str.lower()

In [53]:
temperature = pd.read_csv("temperature_cleaned.csv")
temperature.columns = temperature.columns.str.strip().str.replace(" ", "_").str.lower()


In [54]:
temperature["country"] = temperature["country"].replace({
    "Russian Federation": "Russia",
    "Korea, Rep.": "South Korea",
    "Korea-": "North Korea",
    "Iran, Islamic Rep.": "Iran",
    "Congo, Dem. Rep.": "Congo-Kinshasa",
    "Gambia,": "Gambia, The",
    "Guinea Bissau": "Guinea-Bissau",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Myanmar": "Burma",
    "Slovak Republic": "Slovakia",
    "Syrian Arab Republic": "Syria",
    "Timor Leste": "Timor-Leste",
    "Trinidad And Tobago": "Trinidad and Tobago",
    "Venezuela, Rb": "Venezuela",
    "Yemen, Rep.": "Yemen",
    "Bahamas,": "The Bahamas",
})


In [55]:
merged = pd.merge(
    energy[energy["energy_type"] == "all_energy_types"],
    temperature,
    on=["country", "year"],
    how="left"
)

In [56]:
print(f"Countries: {merged['country'].nunique()}")
print(f"China: {len(merged[merged['country'] == 'China'])} rows")
print(f"India: {len(merged[merged['country'] == 'India'])} rows")

Countries: 230
China: 40 rows
India: 40 rows


In [57]:
merged.to_csv("merged_final.csv", index=False)
print("Saved successfully")

Saved successfully


In [58]:
print(os.path.abspath("merged_final.csv"))


c:\Users\New\OneDrive\Documents\Energy\merged_final.csv


In [59]:
energy = energy[energy["country"] != "Former U.S.S.R."]

In [60]:
merged.to_csv("merged_final.csv", index=False)
print("Saved successfully")

Saved successfully
